# Day 16：LLM API 核心参数调优与 SSE 流式响应 (`stream=True`)

> **🎙️ 讲师旁白**：欢迎来到第二阶段第 16 天！今天我们将从 `project_iterations/day_16` 导入 `GenerationConfig`、`parse_sse_line` 与 `StreamingLLMClient`，在离线 `httpx.MockTransport` 环境下完整演练参数校验、SSE 数据帧解析、打字机流式输出与多角色编排。

## 1. 环境初始化与导入 `project_iterations/day_16` 模块

In [1]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day16_dir = repo_root / "project_iterations" / "day_16"
if str(day16_dir) not in sys.path:
    sys.path.insert(0, str(day16_dir))

os.environ.pop("OPENAI_API_KEY", None)

import httpx
from pydantic import ValidationError
from zhishu_agent import (
    GenerationConfig,
    StreamingLLMClient,
    create_mock_sse_transport,
    parse_sse_line,
)

client = StreamingLLMClient()
print(f"初始化成功！运行模式: {'Mock 离线 SSE 通道' if client.is_mock else 'Live API'}")

初始化成功！运行模式: Mock 离线 SSE 通道


## 2. `GenerationConfig` 核心参数校验（`temperature`, `top_p`, `max_tokens`, `frequency_penalty`）
在发起网络请求前，利用 Pydantic v2 强校验拦截越界参数。

In [2]:
strict_cfg = GenerationConfig(temperature=0.1, top_p=0.85, max_tokens=256, frequency_penalty=0.0)
creative_cfg = GenerationConfig(temperature=1.2, top_p=0.95, max_tokens=512, frequency_penalty=0.5)

print("严谨模式参数:", strict_cfg.to_api_params())
print("创意模式参数:", creative_cfg.to_api_params())

for bad_kwargs in [{"temperature": 2.5}, {"top_p": 0.0}, {"max_tokens": 0}, {"frequency_penalty": 3.0}]:
    try:
        GenerationConfig(**bad_kwargs)
    except ValidationError as exc:
        print(f"成功拦截非法参数 {bad_kwargs}: {exc.errors()[0]['msg']}")

严谨模式参数: {'temperature': 0.1, 'top_p': 0.85, 'max_tokens': 256, 'frequency_penalty': 0.0, 'stream': True}
创意模式参数: {'temperature': 1.2, 'top_p': 0.95, 'max_tokens': 512, 'frequency_penalty': 0.5, 'stream': True}
成功拦截非法参数 {'temperature': 2.5}: Input should be less than or equal to 2
成功拦截非法参数 {'top_p': 0.0}: Input should be greater than 0
成功拦截非法参数 {'max_tokens': 0}: Input should be greater than or equal to 1
成功拦截非法参数 {'frequency_penalty': 3.0}: Input should be less than or equal to 2


## 3. SSE 单行帧解析器 `parse_sse_line` 与 `[DONE]` 终止
验证心跳行、仅含 `role` 的首帧、正文分片与 `data: [DONE]` 的解析行为。

In [3]:
sample_lines = [
    ": keep-alive",
    "",
    "data: {\"choices\":[{\"delta\":{\"role\":\"assistant\"}}]}",
    "data: {\"choices\":[{\"delta\":{\"content\":\"智枢\"}}]}",
    "data: {\"choices\":[{\"delta\":{\"content\":\"平台\"}}]}",
    "data: {\"choices\":[{\"delta\":{\"content\":\"流式就绪！\"}}]}",
    "data: [DONE]",
]

parsed = [parse_sse_line(line) for line in sample_lines]
print("逐行解析结果:", parsed)
print("有效文本聚合:", "".join(item for item in parsed if item is not None))

逐行解析结果: [None, None, None, '智枢', '平台', '流式就绪！', None]
有效文本聚合: 智枢平台流式就绪！


## 4. `StreamingLLMClient` 流式输出、分片聚合与三角色编排
使用 `system`、`user`、`assistant` 三种角色构造上下文，并通过 `MockTransport` 验证流式产出与 `[DONE]` 截断。

In [4]:
custom_transport = create_mock_sse_transport(
    chunks=["【财务合规】", "一线城市住宿上限 ", "600 元/晚，", "二线城市 450 元/晚。"],
    trailing_after_done=["[DONE]之后的脏数据不应出现"],
)
stream_client = StreamingLLMClient(transport=custom_transport)

messages = [
    {"role": "system", "content": "你是智枢企业财务合规顾问。"},
    {"role": "user", "content": "一线城市出差住宿标准是多少？"},
    {"role": "assistant", "content": "一线城市住宿上限为 600 元/晚。"},
    {"role": "user", "content": "二线城市呢？请对比说明。"},
]

chunks = list(stream_client.stream_chat(messages, config=strict_cfg))
print("流式分片列表:", chunks)
print("聚合完整回答:", "".join(chunks))
assert stream_client.chat_aggregated(messages, config=strict_cfg) == "".join(chunks)

流式分片列表: ['【财务合规】', '一线城市住宿上限 ', '600 元/晚，', '二线城市 450 元/晚。']
聚合完整回答: 【财务合规】一线城市住宿上限 600 元/晚，二线城市 450 元/晚。
